# EN3160 - Computer Vision and Image Processing
## Assignment 03: Neural Networks

**Name:** K. D. Manatunga

**Index Number:** 230395T

**GitHub:** https://github.com/KaveeDiv23/Computer-Vision-Assignment-03

In [ ]:
from pathlib import Path
from copy import deepcopy
import random
import time
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display, Markdown

In [ ]:
SEED = 42
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = 0

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Device:", DEVICE)

In [ ]:
def make_loaders(train_data, test_data, batch_size):
    common = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    train = DataLoader(
        train_data, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(SEED), **common
    )
    train_eval = DataLoader(train_data, batch_size=256, shuffle=False, **common)
    test = DataLoader(test_data, batch_size=256, shuffle=False, **common)
    return train, train_eval, test


def plot_history(history, title, eval_name="Test"):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    for ax, metric, label in zip(
        axes, ["loss", "acc"], ["Cross-Entropy Loss", "Accuracy (%)"]
    ):
        ax.plot(df["epoch"], df[f"train_{metric}"], "-o", label="Training")
        ax.plot(df["epoch"], df[f"eval_{metric}"], "-o", label=eval_name)
        ax.set(xlabel="Epoch", ylabel=label, title=label)
        ax.set_xticks(df["epoch"])
        ax.legend()
        ax.grid(alpha=0.3)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def show_cm(y_true, y_pred, class_names, title):
    cm = confusion_matrix(
        y_true, y_pred, labels=range(len(class_names))
    )
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(
        cm, display_labels=class_names
    ).plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm

##### Question 1 - Fully Connected Neural Network

###### Methodology

The provided single-layer network was extended by introducing a
hidden layer containing 100 sigmoid neurons. Each 32 × 32 RGB
CIFAR-10 image was flattened into a 3072-dimensional input vector.
The network produces 10 output logits corresponding to the CIFAR-10
classes.

Multiclass cross-entropy replaced the original mean squared error
loss. Forward propagation, gradient computation, and parameter
updates were implemented manually. The network was trained for
10 epochs and evaluated using training and test accuracy.

In [ ]:
q1_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])

cifar_train = datasets.CIFAR10(
    DATA_DIR, train=True, download=True, transform=q1_transform
)
cifar_test = datasets.CIFAR10(
    DATA_DIR, train=False, download=True, transform=q1_transform
)
q1_train, q1_train_eval, q1_test = make_loaders(
    cifar_train, cifar_test, batch_size=50
)

print("Training samples:", len(cifar_train))
print("Test samples:", len(cifar_test))
print("Classes:", cifar_train.classes)

In [ ]:
DIN, HIDDEN, K = 3072, 100, 10
Q1_EPOCHS, Q1_LR, Q1_DECAY = 10, 0.1, 0.9

set_seed()

w1 = torch.randn(DIN, HIDDEN, device=DEVICE) * np.sqrt(2/(DIN+HIDDEN))
b1 = torch.zeros(HIDDEN, device=DEVICE)
w2 = torch.randn(HIDDEN, K, device=DEVICE) * np.sqrt(2/(HIDDEN+K))
b2 = torch.zeros(K, device=DEVICE)
q1_params = [w1, b1, w2, b2]

def dense_forward(x, params):
    w1, b1, w2, b2 = params
    h = torch.sigmoid(x @ w1 + b1)
    return h, h @ w2 + b2

def manual_backward(x, y, params):
    w1, b1, w2, b2 = params; h, z = dense_forward(x, params)
    log_probs = z - torch.logsumexp(z, dim=1, keepdim=True)
    loss = -log_probs[torch.arange(len(y), device=y.device), y].mean()
    d2 = log_probs.exp()
    d2[torch.arange(len(y), device=y.device), y] -= 1
    d2 /= len(y)
    dw2, db2 = h.T @ d2, d2.sum(0)
    d1 = (d2 @ w2.T) * h * (1-h)
    dw1, db1 = x.T @ d1, d1.sum(0)

    return loss, (dw1, db1, dw2, db2)

print("W1:", w1.shape, "W2:", w2.shape)

In [ ]:
x = torch.randn(4, 5)
y = torch.tensor([0, 1, 0, 1])
p = [torch.randn(5, 3), torch.zeros(3),
     torch.randn(3, 2), torch.zeros(2)]

loss, grads = manual_backward(x, y, p)
p_auto = [v.clone().requires_grad_() for v in p]
_, logits = dense_forward(x, p_auto)
auto_grads = torch.autograd.grad(F.cross_entropy(logits, y), p_auto)

error = max((a-b).abs().max().item() for a, b in zip(grads, auto_grads))
print(f"Maximum gradient difference: {error:.2e}")
assert error < 1e-5